In [6]:
!pip install transformers

In [7]:
# Import libraries
from transformers import AutoTokenizer, AutoModelForTokenClassification
import sqlite3
import torch

In [8]:
# Load the tokenizer and model for BERT-based Indonesian
tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-522M")
model = AutoModelForTokenClassification.from_pretrained("cahya/bert-base-indonesian-522M")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cahya/bert-base-indonesian-522M and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Connect to SQLite database (in-memory)
db = sqlite3.connect(':memory:')

# Create cursor
try:
    cursor = db.cursor()
    print("Cursor created successfully")
except sqlite3.Error as e:
    print(f"Error creating cursor: {e}")
    # Handle the error or exit the program

# Create tables
cursor.execute("CREATE TABLE IF NOT EXISTS texts (id INTEGER PRIMARY KEY AUTOINCREMENT, text TEXT)")
cursor.execute("CREATE TABLE IF NOT EXISTS entities (id INTEGER PRIMARY KEY AUTOINCREMENT, text_id INTEGER, entity TEXT, label TEXT, FOREIGN KEY (text_id) REFERENCES texts(id))")

Cursor created successfully


In [10]:
# Function to add text and entities to the database
def add_text_and_entities(texts):
  for text in texts:
    # Tokenize and get token predictions
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model(**inputs).logits
    predictions = torch.argmax(outputs, dim=2)

    # Extract entities based on token classification
    entities = [(token, model.config.id2label[prediction.item()]) for token, prediction in zip(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]), predictions[0])]

    cursor.execute("INSERT INTO texts (text) VALUES (?)", (text,))
    text_id = cursor.lastrowid
    for token, label in entities:
        cursor.execute("INSERT INTO entities (text_id, entity, label) VALUES (?, ?, ?)", (text_id, token, label))
    db.commit()

# Function to fetch and print all data from the 'texts' table
def print_all_texts():
    cursor.execute("SELECT * FROM texts")
    texts = cursor.fetchall()
    print("Contents of 'texts' table:")
    for row in texts:
        print(row)

# Function to fetch and print all data from the 'entities' table
def print_all_entities():
    cursor.execute("SELECT * FROM entities")
    entities = cursor.fetchall()
    print(entities)
    print("Contents of 'entities' table:")
    for row in entities:
        print(row)

In [11]:
# Example text
texts = [
    # "Presiden Joko Widodo mengunjungi kantor Google di Mountain View, California, Amerika Serikat pada hari Selasa, 25 Februari 2020.",
    "Pada hari Senin, saya pergi ke kantor dengan antusiasme yang tinggi.",
    "Film yang baru dirilis menawarkan cerita yang sangat menarik dan penuh emosi.",
    "Setelah bertahun-tahun bekerja keras, akhirnya saya meraih gelar doktor dalam bidang ilmu komputer.",
    "Pertemuan bisnis penting dilaksanakan di ruang konferensi gedung perkantoran.",
    "Liburan kali ini saya habiskan di pantai, menikmati matahari terbenam yang indah.",
    "Buku terbaru dari penulis terkenal telah menjadi bestseller dalam waktu singkat.",
    "Sebagai seorang ilmuwan, penelitian saya fokus pada pengembangan teknologi ramah lingkungan.",
    "Kegiatan seni dan budaya selalu memberikan pengalaman yang mendalam dan berkesan.",
    "Restoran baru di kota ini menyajikan hidangan khas dengan cita rasa yang unik.",
    "Kompetisi olahraga tahunan akan segera dimulai, dan atlet-atlet terbaik bersiap untuk bertanding.",
    "Di tengah kebisingan kota, taman ini menjadi tempat yang tenang untuk bersantai.",
    "Perusahaan teknologi terkemuka mengumumkan peluncuran produk inovatif terbaru.",
    "Saat musim semi tiba, bunga-bunga di taman bermekaran dengan warna-warna yang cerah.",
    "Acara amal diadakan untuk mengumpulkan dana bagi anak-anak yang membutuhkan.",
    "Pemilihan umum akan segera dilaksanakan, dan calon-calon mulai melakukan kampanye.",
    "Kejuaraan dunia dalam bidang sains dan teknologi menjadi sorotan media internasional.",
    "Penghargaan bergengsi diberikan kepada seniman yang telah memberikan kontribusi besar dalam dunia seni.",
    "Rumah sakit terkemuka meluncurkan program kesehatan masyarakat untuk meningkatkan kesejahteraan.",
    "Pertunjukan teater malam ini akan menampilkan drama klasik yang selalu menginspirasi.",
    "Di tengah pandemi, penelitian medis terus dilakukan untuk menemukan vaksin yang efektif.",
    "Kampus universitas dipenuhi dengan semangat belajar dan kegiatan akademis.",
    "Festival musik tahunan menjadi ajang kumpul para penggemar musik dari berbagai kalangan.",
    "Museum seni modern menampilkan karya seniman kontemporer yang memukau.",
    "Program mentoring diadakan untuk membimbing generasi muda dalam mencapai potensi maksimal.",
    "Perjalanan bisnis kali ini membawa saya ke berbagai kota di seluruh dunia.",
    "Kolaborasi antara perusahaan teknologi dan lembaga pendidikan mempercepat perkembangan inovasi.",
    "Resep masakan tradisional warisan keluarga selalu menjadi favorit di setiap acara keluarga.",
    "Peluncuran satelit baru akan meningkatkan konektivitas internet di daerah terpencil.",
    "Dalam konferensi internasional, para ahli berbagi pengetahuan tentang isu-isu global terkini.",
    "Kegiatan sukarela di komunitas membantu membangun hubungan yang solid di antara warganya.",
    "Pengembangan teknologi kecerdasan buatan menjadi fokus utama dalam industri teknologi.",
    "Program pelatihan keterampilan dilaksanakan untuk meningkatkan daya saing tenaga kerja.",
    "Di tengah cuaca dingin, minuman hangat menjadi pilihan untuk menghangatkan tubuh.",
    "Proyek penanaman pohon di kota bertujuan untuk meningkatkan kualitas udara dan lingkungan.",
    "Selama bulan Ramadan, kegiatan amal dan ibadah menjadi lebih intensif.",
    "Turnamen esports menarik perhatian masyarakat luas dan mendapatkan popularitas yang pesat.",
    "Inovasi dalam industri mode berkontribusi pada pengembangan material ramah lingkungan.",
    "Keberhasilan misi luar angkasa baru-baru ini menandai pencapaian besar dalam eksplorasi angkasa.",
    "Restoran vegetarian yang baru dibuka menyajikan hidangan lezat tanpa bahan hewani.",
    "Penyelidikan jurnalistik mengungkapkan isu-isu sosial yang perlu mendapat perhatian.",
    "Acara pameran seni lokal memberikan platform bagi seniman lokal untuk memamerkan karya mereka.",
    "Perubahan iklim menjadi perhatian utama, dan upaya untuk mengurangi jejak karbon semakin intensif.",
    "Program beasiswa membantu mahasiswa berbakat untuk mengejar pendidikan tinggi.",
    "Sirkuit balap terkenal menjadi tuan rumah untuk kejuaraan mobil balap tahunan.",
    "Teknologi komunikasi terbaru memberikan kemudahan dalam berkomunikasi jarak jauh.",
    "Pameran buku di kota ini menampilkan karya-karya dari penulis lokal dan internasional.",
    "Film dokumenter tentang keberlanjutan lingkungan mendidik masyarakat tentang pentingnya pelestarian alam.",
    "Perguruan tinggi membuka program studi baru untuk memenuhi kebutuhan pasar kerja.",
    "Desa wisata di pedesaan menawarkan pengalaman unik tentang kehidupan desa.",
    "Kolaborasi antara seniman dan ilmuwan menciptakan karya seni yang menggabungkan estetika dan pengetahuan."
]


# Add text and entities to the database
add_text_and_entities(texts)


print_all_texts()
print_all_entities()

Contents of 'texts' table:
(1, 'Pada hari Senin, saya pergi ke kantor dengan antusiasme yang tinggi.')
(2, 'Film yang baru dirilis menawarkan cerita yang sangat menarik dan penuh emosi.')
(3, 'Setelah bertahun-tahun bekerja keras, akhirnya saya meraih gelar doktor dalam bidang ilmu komputer.')
(4, 'Pertemuan bisnis penting dilaksanakan di ruang konferensi gedung perkantoran.')
(5, 'Liburan kali ini saya habiskan di pantai, menikmati matahari terbenam yang indah.')
(6, 'Buku terbaru dari penulis terkenal telah menjadi bestseller dalam waktu singkat.')
(7, 'Sebagai seorang ilmuwan, penelitian saya fokus pada pengembangan teknologi ramah lingkungan.')
(8, 'Kegiatan seni dan budaya selalu memberikan pengalaman yang mendalam dan berkesan.')
(9, 'Restoran baru di kota ini menyajikan hidangan khas dengan cita rasa yang unik.')
(10, 'Kompetisi olahraga tahunan akan segera dimulai, dan atlet-atlet terbaik bersiap untuk bertanding.')
(11, 'Di tengah kebisingan kota, taman ini menjadi tempat yang

In [12]:
# Function to search text by entities
def search_text_by_entitiesV1(entity, label):
    query = "SELECT texts.text FROM texts JOIN entities ON texts.id = entities.text_id WHERE entities.entity = ? AND entities.label = ?"
    cursor.execute(query, (entity.lower(), label))
    results = cursor.fetchall()
    for result in results:
        print(result[0])

# Fungsi untuk mencari teks berdasarkan entitas yang telah diprediksi
def search_text_by_entitiesV2(keyword):
    # Prediksi entitas dari input pengguna
    inputs = tokenizer(keyword, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model(**inputs).logits
    predictions = torch.argmax(outputs, dim=2)

    # Ambil entitas-entitas yang diprediksi
    predicted_entities = [(token, model.config.id2label[prediction.item()]) for token, prediction in zip(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]), predictions[0])]

    # Buat daftar entitas yang ditemukan
    found_entities = []
    for token, label in predicted_entities:
        if label != "O":# Hanya ambil entitas yang terprediksi
          found_entities.append((token.lower(), label))

    # Buat query untuk mencari teks yang memiliki entitas-entitas berdasarkan hasil prediksi
    query = "SELECT DISTINCT  texts.text FROM texts JOIN entities ON texts.id = entities.text_id WHERE entities.entity = ? AND entities.label = ?"

    # Jalankan query dan dapatkan hasilnya
    for entity, label in found_entities:
        cursor.execute(query, (entity, label))
        results = cursor.fetchall()
        # Cetak hasilnya
        # print(f"Search results for entity '{entity}' and label '{label}':")
        for result in results:
            print(result[0])

In [13]:
search_text_by_entitiesV2("sirkuit balap")

Sirkuit balap terkenal menjadi tuan rumah untuk kejuaraan mobil balap tahunan.
Sirkuit balap terkenal menjadi tuan rumah untuk kejuaraan mobil balap tahunan.


In [14]:
# Example search
search_text_by_entitiesV1("Restoran", "LABEL_0")